In [157]:
import os
import uuid
from datetime import datetime
import json

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
import pyspark.sql.functions as F

## Reading Parquet files from the data directory

In [158]:
INBOX_DIR = "/home/jovyan/data/inbox"

spark = SparkSession.builder.appName("Project1").getOrCreate()


def list_parquet_files(inbox_dir: str):
    if not os.path.isdir(inbox_dir):
        return []

    return sorted(
        os.path.join(inbox_dir, name)
        for name in os.listdir(inbox_dir)
        if name.endswith(".parquet")
    )


parquet_files = list_parquet_files(INBOX_DIR)
print(f"Parquet files to read: {len(parquet_files)}")

Parquet files to read: 3


In [159]:
parquet_files

['/home/jovyan/data/inbox/yellow_tripdata_2025-01.parquet',
 '/home/jovyan/data/inbox/yellow_tripdata_2025-02.parquet',
 '/home/jovyan/data/inbox/yellow_tripdata_2025-03.parquet']

## Verifying whether files have been processed

In [160]:
MANIFEST_PATH = "/home/jovyan/state/manifest.json"

if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, "r") as f:
        manifest = json.load(f)
    
    processed_paths = {
        file["path"]
        for batch in manifest.get("batches", [])
        for file in batch.get("processed_files", [])
    }
    parquet_files = [p for p in parquet_files if p not in processed_paths]

In [161]:
parquet_files

['/home/jovyan/data/inbox/yellow_tripdata_2025-03.parquet']

# Mapping

In [162]:
if parquet_files:
    df = spark.read.option("mergeSchema", "true").parquet(*parquet_files).withColumn(
        "source_file", F.input_file_name()
    )
else:
    df = None
    print("No parquet files found; skipping.")

In [163]:

df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------------------------------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|source_file                                                   |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----

In [164]:
df.describe()

DataFrame[summary: string, VendorID: string, passenger_count: string, trip_distance: string, RatecodeID: string, store_and_fwd_flag: string, PULocationID: string, DOLocationID: string, payment_type: string, fare_amount: string, extra: string, mta_tax: string, tip_amount: string, tolls_amount: string, improvement_surcharge: string, total_amount: string, congestion_surcharge: string, Airport_fee: string, cbd_congestion_fee: string, source_file: string]

## Zone lookup

In [165]:
lookup_files = spark.read.option("mergeSchema", "true").parquet(
    f"{INBOX_DIR}/lookup/*.parquet")

zone_lookup = (
    lookup_files
    .filter(F.col("LocationID").isNotNull() & F.col("Zone").isNotNull())
    .select(
        F.col("LocationID").cast("int").alias("LocationID"),
        F.col("Zone").alias("zone_name")
    )
    .dropDuplicates(["LocationID"])
)

In [166]:
zone_lookup.show(5, truncate=False)

+----------+-----------------------+
|LocationID|zone_name              |
+----------+-----------------------+
|1         |Newark Airport         |
|2         |Jamaica Bay            |
|3         |Allerton/Pelham Gardens|
|4         |Alphabet City          |
|5         |Arden Heights          |
+----------+-----------------------+
only showing top 5 rows


## Mapping

In [167]:
df.filter(
    F.col("fare_amount") < 0
).show(10, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------------------------------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|source_file                                                   |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----

In [168]:
trips = df.filter(
    F.col("tpep_pickup_datetime").isNotNull()
    & F.col("tpep_dropoff_datetime").isNotNull()
    & F.col("PULocationID").isNotNull()
    & F.col("DOLocationID").isNotNull()
)

In [169]:
# Keep trip rows only
trips = df.filter(
    (F.col("tpep_pickup_datetime").isNotNull())
    & (F.col("tpep_dropoff_datetime").isNotNull())
    & (F.col("PULocationID").isNotNull())
    & (F.col("DOLocationID").isNotNull())
    & (F.col("passenger_count") >= 0)
    & (F.col("trip_distance") >= 0)
    & (F.col("fare_amount") >= 0)
    & (F.col("tip_amount") >= 0)
)

# Join with trips data
mapped_df = (
    trips.alias("t")
    .join(zone_lookup.alias("pu"), F.col("t.PULocationID") == F.col("pu.LocationID"), "left")
    .join(zone_lookup.alias("do"), F.col("t.DOLocationID") == F.col("do.LocationID"), "left")
    .select(
        F.col("t.tpep_pickup_datetime").alias("pickup_timestamp"),
        F.col("t.tpep_dropoff_datetime").alias("dropoff_timestamp"),
        F.col("t.PULocationID").alias("pickup_location_id"),
        F.col("t.DOLocationID").alias("dropoff_location_id"),
        F.col("pu.zone_name").alias("pickup_zone_name"),
        F.col("do.zone_name").alias("dropoff_zone_name"),
        F.col("t.passenger_count"),
        F.col("t.trip_distance"),
        F.round(
            (
                (
                    F.unix_timestamp(
                        F.col("t.tpep_dropoff_datetime").cast("timestamp"))
                    - F.unix_timestamp(F.col("t.tpep_pickup_datetime").cast("timestamp"))
                ) / 60.0
            ),
            2,
        ).alias("trip_duration_minutes"),
        F.to_date(F.col("t.tpep_pickup_datetime")).alias("pickup_date"),
        F.col("t.source_file"),
        F.current_timestamp().alias("ingested_at"),
    )
)

mapped_df.show(10, truncate=False)

+-------------------+-------------------+------------------+-------------------+----------------------------+----------------------------+---------------+-------------+---------------------+-----------+--------------------------------------------------------------+--------------------------+
|pickup_timestamp   |dropoff_timestamp  |pickup_location_id|dropoff_location_id|pickup_zone_name            |dropoff_zone_name           |passenger_count|trip_distance|trip_duration_minutes|pickup_date|source_file                                                   |ingested_at               |
+-------------------+-------------------+------------------+-------------------+----------------------------+----------------------------+---------------+-------------+---------------------+-----------+--------------------------------------------------------------+--------------------------+
|2025-03-01 00:17:16|2025-03-01 00:25:52|140               |236                |Lenox Hill East             |Upper East S

## Examples of bad rows



In [170]:
bad_trips = df.filter(
    (F.col("tpep_pickup_datetime").isNull())
    | (F.col("tpep_dropoff_datetime").isNull())
    | (F.col("PULocationID").isNull())
    | (F.col("DOLocationID").isNull())
    | (F.col("passenger_count").isNull() | (F.col("passenger_count") < 0))
    | (F.col("trip_distance").isNull() | (F.col("trip_distance") < 0))
    | (F.col("fare_amount").isNull() | (F.col("fare_amount") < 0))
    | (F.col("tip_amount").isNull() | (F.col("tip_amount") < 0))
)
bad_trips.show(10, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------------------------------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|source_file                                                   |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----

## Writing the Output


In [171]:
MANIFEST_PATH = "/home/jovyan/state/manifest.json"

current_batch = {
    "batch_id": str(uuid.uuid4()),
    "created_at": datetime.now().isoformat(),
    "processed_files": [
        {
            "path": p,
            "size": os.path.getsize(p),
            "modified": datetime.fromtimestamp(os.path.getmtime(p)).isoformat(),
        }
        for p in parquet_files
    ],
    "file_size": sum(os.path.getsize(p) for p in parquet_files),
    "row_count": df.count(),
}

if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, "r") as f:
        manifest = json.load(f)
    manifest.setdefault("batches", []).append(current_batch)
else:
    manifest = {"batches": [current_batch]}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

manifest

{'batches': [{'batch_id': 'd5599d36-8538-4c49-9be6-7a3f5e745006',
   'created_at': '2026-03-04T21:02:37.735021',
   'processed_files': [{'path': '/home/jovyan/data/inbox/yellow_tripdata_2025-01.parquet',
     'size': 59158238,
     'modified': '2026-03-04T18:42:10.672486'},
    {'path': '/home/jovyan/data/inbox/yellow_tripdata_2025-02.parquet',
     'size': 60343086,
     'modified': '2026-03-04T18:42:11.099716'}],
   'file_size': 119501324,
   'row_count': 7052769},
  {'batch_id': 'c0b74316-21d3-41c0-a06a-6ef9ba23b1f8',
   'created_at': '2026-03-04T21:06:46.806647',
   'processed_files': [{'path': '/home/jovyan/data/inbox/yellow_tripdata_2025-03.parquet',
     'size': 69964745,
     'modified': '2026-03-04T20:56:16.106220'}],
   'file_size': 69964745,
   'row_count': 4145257}]}

In [172]:
import glob
import shutil

TEMP_DIR = "/home/jovyan/data/outbox/_temp_enriched"
OUTBOX_PATH = "/home/jovyan/data/outbox/trips_enriched.parquet"

if os.path.exists(OUTBOX_PATH):
    existing = spark.read.parquet(OUTBOX_PATH)
    combined = existing.unionByName(mapped_df, allowMissingColumns=True)
else:
    combined = mapped_df

# Write single file
combined.coalesce(1).write.mode("overwrite").parquet(TEMP_DIR)
parquet_file = glob.glob(f"{TEMP_DIR}/*.parquet")[0]
shutil.move(parquet_file, OUTBOX_PATH)
shutil.rmtree(TEMP_DIR)

In [173]:
os.path.getsize(OUTBOX_PATH)

161161076